# MPS Basics and Pauli-Space Conversion

This notebook is a slightly more focused, notebook-first replacement for the old `MPSBasics.jl` script. The goal is to show a complete round trip:

1. start from a dense state vector,
2. convert it to an MPS,
3. convert that state into a Pauli-space object,
4. inspect the bond dimensions that appear along the way.

The calculation is intentionally small so that every step stays transparent.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics

using ITensors
using ITensorMPS


## Build a random state and convert it to an MPS

For a generic state, the MPS bond dimensions encode how much entanglement is needed across each cut. We record those dimensions explicitly because they are often the first diagnostic to look at in tensor-network calculations.

In [ ]:
L = 5
s = siteinds(2, L)
ps = siteinds("Pauli", L)

psi_vec = normalize(randn(ComplexF64, 2^L))
psi_mps = vec2mps(psi_vec, s)
psi_back = mps2vec(psi_mps)

bond_dims = [linkdim(psi_mps, b) for b in 1:length(psi_mps)-1]
summary_state = (
    roundtrip_error = norm(psi_vec - psi_back),
    bond_dims = bond_dims,
    max_bond_dim = maximum(bond_dims; init = 1),
)
summary_state


## Move into Pauli space

`EDKit` also provides conversions between a state MPS, a Pauli-space MPS, and MPO representations. This is useful when you want to reinterpret the same data as an operator object rather than a state vector.

In [ ]:
pmps = mps2pmps(psi_mps, ps)
rho_mpo = pmps2mpo(pmps, s)
pmpo = mpo2pmpo(rho_mpo, ps)

summary_ops = (
    pauli_mps_length = length(pmps),
    density_mpo_length = length(rho_mpo),
    pauli_mpo_length = length(pmpo),
    pauli_mps_bonds = [linkdim(pmps, b) for b in 1:length(pmps)-1],
)
summary_ops


## Plot the MPS bond dimensions

A plot is often easier to read than a raw vector of link dimensions, especially once you start comparing different states or truncation strategies.

In [ ]:
if isdefined(Main, :IJulia)
    using Plots
    bar(1:length(bond_dims), bond_dims; xlabel = "bond", ylabel = "dimension", label = "psi", title = "MPS bond dimensions")
else
    @info "Skipping Plots outside IJulia; returning bond dimensions instead"
    bond_dims
end


## Takeaway

This is the practical core of the old script example: the dense-to-MPS round trip is accurate to machine precision here, and the Pauli-space conversion layer preserves the chain length while changing representation. Once this is familiar, the broader tensor-network examples in `TensorNetworks/Tensors.ipynb` will make much more sense.